# Training ResNet-18 on CIFAR-100
ResNet, short for Residual Networks, is a classic neural network used in the field of deep learning. We will implement the ResNet-18 architecture from scratch and train it on the CIFAR-10 dataset. ResNet-18 is a 18-layer deep network with residual connections.



## 1. Import Libraries

In [ ]:
_ROOT_ = '../'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms


## 2. Define Hyperparameters
Set up the hyperparameters for the training process.

In [ ]:
num_epochs = 200
batch_size = 128

## 3. Load and Normalize CIFAR-100

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR100(root= _ROOT_ + './Data/cifar100', train=True, download=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)

testset = torchvision.datasets.CIFAR100(_ROOT_ + './Data/cifar100', train=False, download=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,shuffle=False)


Visualisation of image samples from CIFAR-100 dataset.

![Architecture](./img/cifar100.png)

## 4. Define ResNet

ResNet, short for Residual Networks, is a type of neural network architecture that enables the training of much deeper networks by leveraging skip connections or shortcuts to jump over some layers. The fundamental building block of a ResNet is the residual block, which adds the input of the block to its output, effectively allowing the network to learn a residual mapping.

Key characteristics of ResNet include:

1. **Residual Blocks:** Each block has a shortcut connection that bypasses one or more layers. In a ResNet, the output of a residual block is computed as the sum of the input and the output of the stacked layers in the block.

2. **Identity Shortcut Connection:** The shortcut connections perform identity mapping, and their outputs are added to the outputs of the stacked layers. If the dimensions are not the same, it can be performed with a linear projection to match the dimensions.

3. **Deeper Network:** By using residual blocks, ResNet can build very deep networks (having 50, 101, or even 152 layers) with improved performance. The skip connections help in addressing the vanishing gradient problem, allowing effective training of deep networks.

4. **Bottleneck Architecture:** For deeper networks, ResNet uses a bottleneck design with three layers in each block. The three layers are 1x1, 3x3, and 1x1 convolutions, where the 1x1 layers are responsible for reducing and then increasing (restoring) dimensions, leaving the 3x3 layer as a bottleneck with smaller input/output dimensions.

5. Improved Training: With residual blocks, gradients can flow directly through the skip connections backward from later layers to earlier layers, making it easier to train deeper networks.

### The diagram below illustrates the architecture of ResNet.
![Architecture](./img/resnet_1.jpeg)

![Architecture](./img/resnet_2.jpeg)

### Create ResNet Architecture

In [ ]:
# Define Convolution
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

# Define the basic block of ResNet
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

# Define the Bottle block of ResNet
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


# Define the ResNet model
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=100, **kwargs):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def _forward_impl(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

    def forward(self, x):
        return self._forward_impl(x)


def resnet18(**kwargs):
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    return model

def resnet34(**kwargs):
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    return model

def resnet50(**kwargs):
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
    return model

def resnet101(**kwargs):
    model = ResNet(Bottleneck, [3, 4, 23, 3], **kwargs)
    return model





## 5. Define a Loss Function and Optimizer

In [ ]:
import torch.optim as optim

# Define the ResNet-18 model
model = resnet18()

# Define the loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

## 6. Train the Network

In [ ]:
from torch.optim.lr_scheduler import MultiStepLR
scheduler = MultiStepLR(optimizer, milestones=[60,120,160], gamma=0.2)

for epoch in range(num_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 0:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 100))
            running_loss = 0.0
    scheduler.step()

print('Finished Training')


## 7. Save the Model
Optionally, save the trained model.

In [ ]:
torch.save(model.state_dict(), './resnet18.pth')

## 8. Test the Model
Test the trained model on the test dataset.

In [ ]:
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (100 * correct / total))


## 9. Validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model = resnet18()

# Load the pretrained weights
model.load_state_dict(torch.load(_ROOT_ + './Model/resnet18-epoch_050.pth'))
model.eval()

# Randomly select an image from the test dataset
dataiter = iter(testloader)
images, labels = next(dataiter)
idx = np.random.randint(images.shape[0])
img, label = images[idx], labels[idx]

# Visualize the image
img = (img * 0.5 + 0.5).clamp(0, 1) # Denomalise the image
img_np = img.numpy()
plt.imshow(np.transpose(img_np, (1, 2, 0)))
plt.title(f'Actual label: {label}')
plt.show()

# Predict with the model
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Retrieve the CIFAR-100 classes
classes = testset.classes

# Print the model's predicted and actual labels
print(f'Predicted: {classes[predicted[0]]}, Actual: {classes[label]}')

## References:
1. CIFAR-100 Dataset: Krizhevsky, A., & Hinton, G. (2009). ["Learning multiple layers of features from tiny images"](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf).

2. PyTorch Documentation: Accessible at [PyTorch Docs](https://pytorch.org/docs/stable/index.html).

3. Torchvision Models: Detailed in the [Torchvision Models Documentation](https://pytorch.org/vision/stable/models.html).

4. PyTorch Vision Models: [PyTorch Vision Model Zoo](https://pytorch.org/vision/stable/models.html)


